# Inner vs Left Join

<a href="https://colab.research.google.com/github/vuhung16au/ACU-ITEC102/blob/main/Week09/03.Inner-vs-Left-Join/notebooks/01_03.Inner-vs-Left-Join.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Overview
When merging relational tables, the **join type** (`how` parameter) dictates which records are preserved and which are discarded:
- **Inner Join (`how='inner'`)**: Keeps only records where the key exists in **both** tables (intersection).
- **Left Join (`how='left'`)**: Keeps **all** records from the left table, filling unmatched right-table fields with `NaN`.
- **Right Join (`how='right'`)**: Keeps all records from the right table.
- **Outer Join (`how='outer'`)**: Keeps all records from both tables (union).

In business and data science analytics, **Left Joins are the standard default** because you rarely want primary entities (customers, patients, students) to disappear simply because they lack an optional secondary record.

## 1. Setup: Students and Scholarships Tables
Construct a primary Students table and an auxiliary Scholarships table.

In [ ]:
import pandas as pd

students = pd.DataFrame({
    'StudentID': [101, 102, 103, 104, 105],
    'Name': ['Liam Nguyen', 'Emma Watson', 'Oliver Brown', 'Sophia Vu', 'Noah Taylor'],
    'Campus': ['Sydney', 'Melbourne', 'Brisbane', 'Sydney', 'Perth']
})

scholarships = pd.DataFrame({
    'StudentID': [102, 104, 106, 107],
    'ScholarshipName': ['Vice-Chancellor Award', 'STEM Excellence', 'Alumni Grant', 'Regional Bursary'],
    'Amount_AUD': [5000, 7500, 3000, 4000]
})

display(students)
display(scholarships)

## 2. Inner Join (how='inner'): Intersection Only
Only records with matching keys in **both** tables are included. Students without scholarships (101, 103, 105) and awards for non-current students (106, 107) are dropped.

In [ ]:
inner_res = pd.merge(students, scholarships, on='StudentID', how='inner')
print(f"Row count: {len(inner_res)}")
display(inner_res)

## 3. Left Join (how='left'): Preserving All Primary Entities
Every single student from the left table is retained. Students without scholarships receive `NaN` for scholarship columns.

In [ ]:
left_res = pd.merge(students, scholarships, on='StudentID', how='left')
print(f"Row count: {len(left_res)} (matches left table length)")
display(left_res)

## 4. Identifying Non-Matching Entities (Left Join + isna)
Find which students do not receive any scholarship by filtering where the right-side columns are null.

In [ ]:
unfunded = left_res[left_res['ScholarshipName'].isna()]
print("Students with no scholarship:")
display(unfunded[['StudentID', 'Name', 'Campus']])

## 5. Full Outer Join (how='outer')
Retains all records from both datasets, with `NaN` wherever a match does not exist.

In [ ]:
outer_res = pd.merge(students, scholarships, on='StudentID', how='outer')
display(outer_res)

## Enrichment
### Preventing Row Multiplication with validate
```python
# Guard against unexpected duplicates in key columns:
pd.merge(students, scholarships, on='StudentID', how='left', validate='one_to_one')
```

## Takeaways
- **Inner Join**: Intersection of keys. Unmatched rows on either side are silently deleted.
- **Left Join**: Preserves 100% of rows from the left table. Unmatched right columns become `NaN`.
- In data reporting, default to **Left Join** so primary entities (students, customers) are never lost.
- Combine Left Join with `.isna()` to quickly discover non-matching / unengaged entities.

## Conclusion
Selecting the right join strategy is critical to avoid accidental data deletion and prevent false assumptions in downstream analytics.

## Exercises
**Exercise 1:** Calculate how many students were dropped by the inner join compared to the total student count.

**Exercise 2:** Calculate the total scholarship dollars awarded to currently active students using `inner_res`.

**Exercise 3:** Perform a Right Join (`how='right'`) on `students` and `scholarships`. Which scholarship award has no matching active student?

In [ ]:
# Write your practice code here

# --- Solutions ---
# dropped = len(students) - len(inner_res)
# print(f"Dropped students: {dropped}")
#
# total_funding = inner_res['Amount_AUD'].sum()
# print(f"Total funding: ${total_funding:,} AUD")
#
# right_res = pd.merge(students, scholarships, on='StudentID', how='right')
# display(right_res[right_res['Name'].isna()])